# Africa Disease Burden Analysis — Interactive Dashboard (2000–2021)
**Author:** Afriyie Karikari Bempah, PharmD  
**Data:** WHO Global Health Observatory API (live data)  
**Tools:** Python, requests, pandas, Plotly

---

### Research Questions
1. How does life expectancy vary across 47 African countries?
2. Which countries have improved most over 20 years — and why?
3. How does Ghana compare to regional peers?
4. What is the relationship between child mortality and life expectancy?

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Pull WHO Data via API

In [2]:
# Life expectancy data
url_le = "https://ghoapi.azureedge.net/api/WHOSIS_000001"
response_le = requests.get(url_le)
df_raw = pd.DataFrame(response_le.json()['value'])

# Under-5 mortality data
url_mort = "https://ghoapi.azureedge.net/api/MDG_0000000007"
response_mort = requests.get(url_mort)
df_mort_raw = pd.DataFrame(response_mort.json()['value'])

# Country names
url_countries = "https://ghoapi.azureedge.net/api/DIMENSION/COUNTRY/DimensionValues"
response_countries = requests.get(url_countries)
df_countries = pd.DataFrame(response_countries.json()['value'])[['Code', 'Title']]
df_countries.columns = ['Country_Code', 'Country_Name']

print(f"Life expectancy records: {len(df_raw):,}")
print(f"Mortality records: {len(df_mort_raw):,}")
print(f"Countries in reference: {len(df_countries):,}")

Life expectancy records: 12,936
Mortality records: 63,070
Countries in reference: 234


## 2. Filter and Clean Data

In [3]:
# filter life expectancy to Africa, both sexes
africa = df_raw[
    (df_raw['ParentLocation'] == 'Africa') &
    (df_raw['Dim1'] == 'SEX_BTSX')
][['SpatialDim', 'TimeDim', 'NumericValue']].copy()
africa.columns = ['Country_Code', 'Year', 'Life_Expectancy']
africa = africa.dropna(subset=['Life_Expectancy'])

# filter mortality to Africa
df_mort = df_mort_raw[
    (df_mort_raw['ParentLocation'] == 'Africa') &
    (df_mort_raw['Dim1'] == 'SEX_BTSX')
][['SpatialDim', 'TimeDim', 'NumericValue']].copy()
df_mort.columns = ['Country_Code', 'Year', 'Under5_Mortality']
df_mort = df_mort.dropna(subset=['Under5_Mortality'])

# merge and deduplicate
df_combined = africa.merge(
    df_mort[['Country_Code', 'Year', 'Under5_Mortality']],
    on=['Country_Code', 'Year'], how='inner'
)
df_combined = df_combined.groupby(['Country_Code', 'Year']).agg(
    {'Life_Expectancy': 'mean', 'Under5_Mortality': 'mean'}
).reset_index()
df_combined = df_combined.merge(df_countries, on='Country_Code', how='left')

print(f"Final dataset: {df_combined.shape}")
print(f"Countries: {df_combined['Country_Code'].nunique()}")
print(f"Year range: {df_combined['Year'].min()} to {df_combined['Year'].max()}")

Final dataset: (1034, 5)
Countries: 47
Year range: 2000 to 2021


## 3. Life Expectancy Trends — Top and Bottom 5 Countries

In [4]:
latest = df_combined[df_combined['Year'] == 2021].sort_values('Life_Expectancy', ascending=False)
top5 = latest.head(5)['Country_Code'].tolist()
bottom5 = latest.tail(5)['Country_Code'].tolist()

df_highlight = df_combined[df_combined['Country_Code'].isin(top5 + bottom5)]

fig = px.line(
    df_highlight,
    x='Year', y='Life_Expectancy',
    color='Country_Name',
    title='Life Expectancy Trends — Top 5 and Bottom 5 African Countries (2000–2021)',
    labels={'Life_Expectancy': 'Life Expectancy (Years)'}
)
fig.update_layout(height=500)
fig.show()

print("Top 5 countries (2021):")
print(latest[['Country_Name', 'Life_Expectancy']].head(5).round(1).to_string())
print("\nBottom 5 countries (2021):")
print(latest[['Country_Name', 'Life_Expectancy']].tail(5).round(1).to_string())

# Insight: North African and island nations lead life expectancy.
# HIV/AIDS epidemic explains the dip in Southern African countries pre-2005.

Top 5 countries (2021):
              Country_Name  Life_Expectancy
285                Algeria             76.0
879             Seychelles             74.0
637              Mauritius             73.4
263             Cabo Verde             73.2
835  Sao Tome and Principe             71.2

Bottom 5 countries (2021):
                  Country_Name  Life_Expectancy
1033                  Zimbabwe             58.5
593                 Mozambique             57.7
857                   Eswatini             54.6
131   Central African Republic             52.3
527                    Lesotho             51.5


## 4. Choropleth Map — Life Expectancy Across Africa (2021)

In [5]:
latest_2021 = df_combined[df_combined['Year'] == 2021]

fig_map = px.choropleth(
    latest_2021,
    locations='Country_Code',
    color='Life_Expectancy',
    hover_name='Country_Name',
    color_continuous_scale='RdYlGn',
    scope='africa',
    title='Life Expectancy Across Africa (2021)',
    labels={'Life_Expectancy': 'Life Expectancy (Years)'}
)
fig_map.update_layout(height=500)
fig_map.show()

# Insight: Clear North-South divide — North Africa leads with 70-76 years.
# Central and Southern Africa bear the heaviest burden at 52-60 years.

## 5. Ghana Deep Dive

In [6]:
ghana = df_combined[df_combined['Country_Code'] == 'GHA'].copy()

fig_ghana = make_subplots(rows=1, cols=2,
    subplot_titles=('Life Expectancy', 'Under-5 Mortality Rate'))

fig_ghana.add_trace(
    go.Scatter(x=ghana['Year'], y=ghana['Life_Expectancy'],
               mode='lines+markers', name='Life Expectancy',
               line=dict(color='green', width=2)), row=1, col=1)

fig_ghana.add_trace(
    go.Scatter(x=ghana['Year'], y=ghana['Under5_Mortality'],
               mode='lines+markers', name='Under-5 Mortality',
               line=dict(color='red', width=2)), row=1, col=2)

fig_ghana.update_layout(title='Ghana Health Indicators (2000–2021)', height=450)
fig_ghana.show()

print("Ghana Summary (2017-2021):")
print(ghana[['Year', 'Life_Expectancy', 'Under5_Mortality']].tail(5).round(1).to_string())

# Insight: Ghana gained 7 years of life expectancy between 2000-2021.
# Under-5 mortality fell 60% — from 100 to 40 per 1,000 live births.
# Reflects NHIS (2003), expanded primary healthcare, and malaria interventions.

Ghana Summary (2017-2021):
     Year  Life_Expectancy  Under5_Mortality
369  2017             65.1              48.1
370  2018             65.5              46.0
371  2019             66.0              43.9
372  2020             66.2              42.0
373  2021             66.1              40.2


## 6. Scatter — Life Expectancy vs Child Mortality (2021)

In [7]:
fig_scatter = px.scatter(
    latest_2021,
    x='Under5_Mortality', y='Life_Expectancy',
    hover_name='Country_Name',
    size='Under5_Mortality',
    color='Life_Expectancy',
    color_continuous_scale='RdYlGn',
    title='Life Expectancy vs Under-5 Mortality — Africa (2021)',
    labels={'Under5_Mortality': 'Under-5 Mortality (per 1,000)',
            'Life_Expectancy': 'Life Expectancy (Years)'}
)

ghana_2021 = latest_2021[latest_2021['Country_Code'] == 'GHA'].iloc[0]
fig_scatter.add_annotation(
    x=ghana_2021['Under5_Mortality'], y=ghana_2021['Life_Expectancy'],
    text='Ghana', showarrow=True, arrowhead=2,
    font=dict(size=12, color='black')
)
fig_scatter.update_layout(height=500)
fig_scatter.show()

## 7. Top Improvers — Life Expectancy Gained (2000–2021)

In [8]:
progress = df_combined[df_combined['Year'].isin([2000, 2021])]
progress = progress.groupby(['Country_Code', 'Country_Name', 'Year'])[
    'Life_Expectancy'].mean().reset_index()
progress_wide = progress.pivot(
    index=['Country_Code', 'Country_Name'],
    columns='Year', values='Life_Expectancy'
).reset_index()
progress_wide.columns = ['Country_Code', 'Country_Name', 'LE_2000', 'LE_2021']
progress_wide['Improvement'] = progress_wide['LE_2021'] - progress_wide['LE_2000']
progress_wide = progress_wide.sort_values('Improvement', ascending=False)

fig_progress = px.bar(
    progress_wide.head(15),
    x='Country_Name', y='Improvement',
    color='Improvement', color_continuous_scale='Greens',
    title='Top 15 African Countries by Life Expectancy Improvement (2000–2021)',
    labels={'Improvement': 'Years Gained', 'Country_Name': 'Country'}
)
fig_progress.update_layout(height=500, xaxis_tickangle=45)
fig_progress.show()

print("Top 10 improvers:")
print(progress_wide[['Country_Name','LE_2000','LE_2021','Improvement']].head(10).round(1).to_string())

# Insight: Rwanda gained 20.6 years — among the fastest improvements in
# recorded public health history. Reflects post-genocide reconstruction,
# universal health coverage, and Africa's strongest CHW program.
# Ghana's absence from top improvers reflects its stronger 2000 baseline.

Top 10 improvers:
                   Country_Name  LE_2000  LE_2021  Improvement
33                       Rwanda     46.9     67.5         20.6
1                       Burundi     44.4     64.0         19.6
29                       Malawi     44.6     62.5         17.8
43                       Uganda     48.9     66.0         17.1
14                     Ethiopia     50.8     67.8         17.1
45                       Zambia     44.5     61.0         16.4
4                      Botswana     47.2     61.2         14.1
42  United Republic of Tanzania     52.8     66.8         14.0
0                        Angola     49.4     62.1         12.8
46                     Zimbabwe     45.7     58.5         12.7


## 8. Conclusions

| Finding | Implication |
|---|---|
| **North-South divide is stark** | Algeria/Mauritius at 75+ years vs CAR at 52 years — a 23-year gap |
| **Rwanda gained 20.6 years in 21 years** | Community health workers + UHC = transformative impact |
| **Ghana gained 7 years, cut child mortality 60%** | NHIS and primary care expansion visible in data |
| **HIV/AIDS treatment scale-up is visible** | Zimbabwe, Malawi, Zambia all show sharp upturns post-2005 |
| **Child mortality strongly predicts life expectancy** | Investing in child survival has outsized population health returns |

---

**Data Source:** WHO Global Health Observatory API  
**Analysis by:** Afriyie Karikari Bempah, PharmD | [LinkedIn](https://linkedin.com/in/afriyiekarikaribempah) | [GitHub](https://github.com/akbempah1)